In [1]:
from source_code.itransformer_dataset import load_and_preprocess_data

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, rmse

# --- Step A: Load Data ---
df_long, hist_exog_cols, TARGETS = load_and_preprocess_data("/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_planck_weather_ts.csv")

# --- Step B: Train-Test Split (Long Format) ---
HORIZON = 144  # Prediksi 24 timestep ke depan (4 jam)
INPUT_SIZE = 288  # 96 timestep ke belakang (16 jam)
NUM_TARGETS = len(TARGETS)
# Ambil 10.000 titik waktu terakhir, untuk menghemat sumber daya, karena jika data dipakai semua maka itu akan banyak sekali
NUM_TIMESTAMPS = 10000  
TEST_TIMESTAMPS = 2000  # 2.000 titik waktu untuk pengujian

# Ambil 140.000 baris terakhir (10.000 timestamp x 14 fitur)
df_sample = df_long.tail(NUM_TIMESTAMPS * NUM_TARGETS).reset_index(drop=True)

# Cari titik batas pemisah (split) berdasarkan timestamp ds
unique_timestamps = df_sample['ds'].unique()
last_train_ds = unique_timestamps[-TEST_TIMESTAMPS] # tanggal 2000 pertama

train_df = df_sample[df_sample['ds'] < last_train_ds].copy()
test_df = df_sample[df_sample['ds'] >= last_train_ds].copy()

print(
    f'\nInformasi Dataset:\n'
    f'- Jumlah Fitur Target : {NUM_TARGETS} fitur\n'
    f'- Timestep Train Data  : {len(train_df) // NUM_TARGETS} baris per fitur\n'
    f'- Timestep Test Data   : {len(test_df) // NUM_TARGETS} baris per fitur'
)

# --- Step C: Inisialisasi Model iTransformer ---
print('\n2. Menginisialisasi Model iTransformer...')

models = [
    iTransformer(
        h=HORIZON,
        input_size=INPUT_SIZE,
        n_series=NUM_TARGETS,
        max_steps=1000,  # Ubah ke 1000+ jika ingin hasil optimal
        batch_size=32,
        learning_rate=0.001,
        # -----------------------------------------------------------------
        # OPTIMASI VALIDASI:
        val_check_steps=50,  # 👈 Validasi hanya dilakukan setiap 50 steps
        early_stop_patience_steps=3,  # Toleransi 3x pengecekan validasi (3 x 50 = 150 steps)
        # -----------------------------------------------------------------
        scaler_type='standard',  # Normalisasi data otomatis per unique_id
        accelerator ='auto',  # Otomatis deteksi GPU/CPU
    )
]

# Inisialisasi runner NeuralForecast
nf = NeuralForecast(models=models, freq='10m')

# --- Step D: Fit Model ---
VAL_SIZE = 144

print('\n3. Memulai proses Training (Fit)...')
nf.fit(df=train_df, val_size = VAL_SIZE)

# --- Step D.1: Menyimpan Model yang Sudah Dilatih ---
print('\n3.b. Menyimpan model iTransformer...')

# Tentukan folder lokasi penyimpanan
MODEL_PATH = '/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_plank_weather/all_models_training'

# Simpan model (overwrite=True agar menimpa model lama jika ada)
nf.save(path=MODEL_PATH, overwrite=True)

print(f'✅ Model berhasil disimpan di folder: {MODEL_PATH}')

# --- Step E: Cross-Validation / Predict ---
print('\n4. Melakukan Prediksi pada Data Uji (14 Fitur)...')

# Hitung jumlah window prediksi (2000 / 24 = 83 windows)
N_WINDOWS = TEST_TIMESTAMPS // HORIZON

cv_df = nf.cross_validation(
    df=df_sample,
    n_windows=N_WINDOWS,
    test_size=None,  # 👈 Kosongkan test_size secara eksplisit
    val_size=VAL_SIZE,
    use_fitted=True,  # Gunakan model yang sudah di-fit pada Step D
)

print('\nHasil Prediksi (5 baris pertama):')
print(cv_df.head())

# --- Step F: Evaluasi Metrik (MAE & RMSE) ---
print('\n5. Menghitung Metrik Evaluasi per Fitur...')

# 1. Jalankan evaluasi per fitur seperti biasa
evaluation_results = evaluate(
    df=cv_df, metrics=[mae, rmse], models=['iTransformer']
)

# 2. Pivot agar 1 fitur = 1 baris, dengan kolom MAE dan RMSE di sampingnya
feature_scores = evaluation_results.pivot(
    index='unique_id', columns='metric', values='iTransformer'
).reset_index()

# Rapikan nama indeks kolom
feature_scores.columns.name = None

print('=== SKOR PREDIKSI PER FITUR (14 FITUR) ===')
# Urutkan dari fitur yang MAE-nya paling tinggi (paling sulit diprediksi)
feature_scores_sorted = feature_scores.sort_values(
    by='mae', ascending=False
)
print(feature_scores_sorted)

# --- Step G: Visualisasi Hasil Prediksi (Grid 7x2) ---
print('\n6. Menampilkan Grafik Hasil Prediksi 14 Fitur...')

# Ambil 200 titik waktu terakhir per fitur untuk visualisasi
plot_df = cv_df.groupby('unique_id').tail(200)

fig, axes = plt.subplots(7, 2, figsize=(16, 22), sharex=True)
axes = axes.flatten()

for idx, target_name in enumerate(TARGETS):
  sub_df = plot_df[plot_df['unique_id'] == target_name]
  ax = axes[idx]

  ax.plot(
      sub_df['ds'], sub_df['y'], label='Nilai Asli', color='black', alpha=0.8
  )
  ax.plot(
      sub_df['ds'],
      sub_df['iTransformer'],
      label='Prediksi iTransformer',
      color='red',
      linestyle='--',
  )

  ax.set_title(f'Fitur: {target_name}', fontsize=10, fontweight='bold')
  ax.grid(True, linestyle=':', alpha=0.6)
  if idx == 0:
    ax.legend()

plt.suptitle(
    'iTransformer: Hasil Prediksi 14 Fitur Cuaca (Max Planck)',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('hasil_prediksi_14_fitur_itransformer.png')
print(
    'Grafik berhasil disimpan sebagai "hasil_prediksi_14_fitur_itransformer.png"'
)
plt.show()

/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-17 08:11:19,606	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-08-17 08:11:21,181	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


1. Membaca dan memproses dataset multivariate...


Seed set to 1



Informasi Dataset:
- Jumlah Fitur Target : 14 fitur
- Timestep Train Data  : 8000 baris per fitur
- Timestep Test Data   : 2000 baris per fitur

2. Menginisialisasi Model iTransformer...

3. Memulai proses Training (Fit)...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores

  | Name                | Type                   | Params | Mode 
-----------------------------------------------------------------------
0 | loss                | MAE                    | 0      | train
1 | hist_cat_embeddings | ModuleList             | 0      | train
2 | futr_cat_embeddings | ModuleList             | 0      | train
3 | stat_cat_embeddings | ModuleList             | 0      | train
4 | padder_train        | ConstantPad1d          | 0      | train
5 | scaler              | TemporalNorm           | 0      | train
6 | enc_embedding       | DataEmbedding_inverted | 147 K  | train
7 | encoder             | TransEncoder           | 6.3 M  | train
8 | projector           | Linear                 | 73.9 K | train
-----------------------------------------------------------------------
6.5 M     Trainable params
0         Non-trainable params
6.5 M     Total params
26.111    Total estimated model params 

...

/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/myenv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Epoch 399: 100%|██████████| 1/1 [00:00<00:00,  3.02it/s, v_num=8, train_loss_step=1.160, train_loss_epoch=1.160, valid_loss=5.960]

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/myenv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.




3.b. Menyimpan model iTransformer...
✅ Model berhasil disimpan di folder: /workspaces/Max-Plank-Weather-using-iTransformer-PatchTST-DLinear-N-HiTS-dan-GRU/max_plank_weather/all_models_training

4. Melakukan Prediksi pada Data Uji (14 Fitur)...
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 11.78it/s]

Hasil Prediksi (5 baris pertama):
         unique_id                  ds              cutoff  iTransformer     y
0  H2OC (mmol/mol) 2016-12-30 22:10:00 2016-12-30 22:00:00      4.668582  4.04
1  H2OC (mmol/mol) 2016-12-30 22:20:00 2016-12-30 22:00:00      4.710747  4.17
2  H2OC (mmol/mol) 2016-12-30 22:30:00 2016-12-30 22:00:00      4.649713  4.32
3  H2OC (mmol/mol) 2016-12-30 22:40:00 2016-12-30 22:00:00      4.604323  4.37
4  H2OC (mmol/mol) 2016-12-30 22:50:00 2016-12-30 22:00:00      4.706229  4.35

5. Menghitung Metrik Evaluasi per Fitur...


ValueError: Index contains duplicate entries, cannot reshape

In [ ]:
df_long.head()

,ds,day_sin,day_cos,month_sin,month_cos,unique_id,y
0,2009-01-01 00:10:00,0.043619,0.999048,0.5,0.866025,H2OC (mmol/mol),3.12
1,2009-01-01 00:10:00,0.043619,0.999048,0.5,0.866025,T (degC),-8.02
2,2009-01-01 00:10:00,0.043619,0.999048,0.5,0.866025,Tdew (degC),-8.90
3,2009-01-01 00:10:00,0.043619,0.999048,0.5,0.866025,Tpot (K),265.40
4,2009-01-01 00:10:00,0.043619,0.999048,0.5,0.866025,VPact (mbar),3.11
